# Project 1: Thermal Sensation — Compare models and evaluate

In this notebook we will compare a mean baseline, Linear Regression, and Random Forest. We will use three-fold group cross-validation on training buildings to choose a model, then evaluate the chosen model once on the held-out test buildings.

The final test score is an estimate of performance on buildings the model did not train on. Do not use this score to keep changing the model.

## 1. Load the cleaned data and saved split

Run Notebooks 01 and 02 first. They create the cleaned Project 1 data and assign each building to either training or test.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor

project_folder = Path.cwd()
if not (project_folder / "data" / "processed").exists():
    project_folder = project_folder.parent

processed_folder = project_folder / "data" / "processed"
cleaned_data = pd.read_csv(
    processed_folder / "thermal_sensation_cleaned.csv",
    dtype={"building_id": "string", "subject_id": "string"},
    low_memory=False,
)
split_assignments = pd.read_csv(
    processed_folder / "p1_building_split.csv",
    dtype={"building_id": "string"},
)

model_data = cleaned_data.merge(
    split_assignments,
    on="building_id",
    how="left",
    validate="many_to_one",
)

if model_data["data_split"].isna().any():
    raise ValueError("Some building IDs are missing from the saved split.")

training_data = model_data.loc[model_data["data_split"] == "train"].copy()
test_data = model_data.loc[model_data["data_split"] == "test"].copy()

print("Training rows:", len(training_data))
print("Test rows:", len(test_data))

## 2. Set up the model inputs and preprocessing

Use the same initial input columns chosen on Day 2. Missing numeric values are filled with a median, and missing categories with the most common value. These steps are fitted separately inside each cross-validation fold.

In [ ]:
numeric_features = ["ta", "rh", "vel", "clo", "met"]
categorical_features = [
    "season",
    "country",
    "climate",
    "building_type",
    "cooling_type",
]
selected_features = numeric_features + categorical_features
target_column = "thermal_sensation"

X_train = training_data[selected_features]
y_train = training_data[target_column]
training_groups = training_data["building_id"]
X_test = test_data[selected_features]
y_test = test_data[target_column]

numeric_preprocessing = Pipeline(
    steps=[
        ("fill_missing", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
categorical_preprocessing = Pipeline(
    steps=[
        ("fill_missing", SimpleImputer(strategy="most_frequent")),
        ("make_indicators", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numbers", numeric_preprocessing, numeric_features),
        ("categories", categorical_preprocessing, categorical_features),
    ]
)

print("Input features:", selected_features)
print("Training buildings:", training_groups.nunique())

## 3. Compare models using training buildings only

We will compare the mean baseline, Linear Regression, and Random Forest using three-fold group cross-validation. Each fold holds out complete buildings from the training set. Three folds keep this first model comparison manageable while still testing generalization across different buildings.

The model with the lowest average cross-validation MAE will be selected. The test data is not used in this choice.

In [ ]:
models = {
    "Mean baseline": Pipeline(
        steps=[
            ("prepare_features", preprocessor),
            ("regression", DummyRegressor(strategy="mean")),
        ]
    ),
    "Linear Regression": Pipeline(
        steps=[
            ("prepare_features", preprocessor),
            ("regression", LinearRegression()),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("prepare_features", preprocessor),
            (
                "regression",
                RandomForestRegressor(
                    n_estimators=50,
                    random_state=42,
                    n_jobs=1,
                ),
            ),
        ]
    ),
}

group_cross_validation = GroupKFold(n_splits=3)
results = []

for model_name, model in models.items():
    # Scores are negative so that scikit-learn can treat every score as "higher is better".
    negative_mae_scores = cross_val_score(
        model,
        X_train,
        y_train,
        groups=training_groups,
        cv=group_cross_validation,
        scoring="neg_mean_absolute_error",
        n_jobs=1,
    )

    mae_scores = -negative_mae_scores
    results.append(
        {
            "model": model_name,
            "average_MAE": mae_scores.mean(),
            "MAE_standard_deviation": mae_scores.std(),
        }
    )

model_comparison = pd.DataFrame(results).sort_values("average_MAE")
display(model_comparison)

# Keep the mean baseline as a reference; select between models that use features.
feature_model_results = model_comparison[
    model_comparison["model"] != "Mean baseline"
]
best_model_name = feature_model_results.iloc[0]["model"]
best_model = models[best_model_name]
print("Model selected using training cross-validation:", best_model_name)

## 4. Evaluate the selected model on the held-out buildings

Now that the model choice is made using training data only, fit it using all training rows and evaluate it on the test buildings once. MAE and RMSE describe the typical prediction error in sensation-vote units. R² describes how much target variation the model explains relative to a simple average prediction.

In [ ]:
best_model.fit(X_train, y_train)
test_predictions = best_model.predict(X_test)

test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = mean_squared_error(y_test, test_predictions) ** 0.5
test_r2 = r2_score(y_test, test_predictions)

print(f"Test MAE:  {test_mae:.3f}")
print(f"Test RMSE: {test_rmse:.3f}")
print(f"Test R-squared: {test_r2:.3f}")

## 5. Inspect prediction errors and save the model

A residual is the true vote minus the model prediction. Plotting residuals helps show whether errors are centered around zero and whether the model makes larger errors for particular sensation values.

We save the selected fitted pipeline so the next notebook can explain its predictions.

In [ ]:
test_results = pd.DataFrame(
    {
        "building_id": test_data["building_id"].to_numpy(),
        "actual_sensation": y_test.to_numpy(),
        "predicted_sensation": test_predictions,
    }
)
test_results["residual"] = (
    test_results["actual_sensation"] - test_results["predicted_sensation"]
)

plt.figure(figsize=(7, 5))
plt.scatter(test_results["actual_sensation"], test_results["residual"], alpha=0.2)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Actual thermal sensation")
plt.ylabel("Residual (actual - predicted)")
plt.title("Prediction errors on held-out buildings")
plt.show()

outputs_folder = project_folder / "outputs"
outputs_folder.mkdir(parents=True, exist_ok=True)

model_path = outputs_folder / "p1_thermal_sensation_model.joblib"
predictions_path = outputs_folder / "p1_test_predictions.csv"
joblib.dump(best_model, model_path)
test_results.to_csv(predictions_path, index=False)

print("Saved fitted model to:", model_path)
print("Saved test predictions to:", predictions_path)

## Interpretation note

A good test score does not prove that changing the room temperature or fan speed will cause a particular sensation. This is observational data from many studies. We can describe predictive patterns, but causal claims require a different study design.